# GLM-5 Reasoning Distillation into Qwen3.5-4B
## SFT Training with Unsloth QLoRA

This notebook fine-tunes Qwen3.5-4B on reasoning traces generated by GLM-5.
Run this on Google Colab with a T4 GPU (free tier).

In [ ]:
# Step 1: Install Unsloth and dependencies
!pip install unsloth
!pip install trl datasets huggingface_hub

In [ ]:
# Step 2: Login to HuggingFace
from huggingface_hub import login
from google.colab import userdata

# Set your HF token in Colab secrets (key: HF_TOKEN)
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Step 3: Load base model with QLoRA
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

max_seq_length = 4096  # Reasoning traces can be long
dtype = None  # Auto-detect
load_in_4bit = True  # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-4B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
# Step 4: Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # LoRA rank - higher for reasoning patterns
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory efficient
    random_state=42,
)

In [ ]:
# Step 5: Load the reasoning dataset from HuggingFace
from datasets import load_dataset

dataset = load_dataset("bmeyer2025/glm5-reasoning-traces")
print(f"Train: {len(dataset['train'])} examples")
print(f"Test: {len(dataset['test'])} examples")
print()
print("Sample:")
print(dataset['train'][0]['messages'][1]['content'][:200])

In [ ]:
# Step 6: Format dataset with chat template
def formatting_func(examples):
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)
print("Formatted sample:")
print(dataset['train'][0]['text'][:500])

In [ ]:
# Step 7: Configure and run SFT training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch = 8
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
)

# Enable train_on_responses_only so loss is only on assistant turns
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Verify masking is working
# Labels should show -100 for system/user tokens, actual tokens for assistant
space = tokenizer(" ", add_special_tokens=False).input_ids[0]
sample_labels = trainer.train_dataset[0]["labels"]
decoded = tokenizer.decode([space if x == -100 else x for x in sample_labels])
print("Masked training sample (spaces = masked):")
print(decoded[:500])

In [ ]:
# Step 8: Train!
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# Step 9: Test generation on unseen problems
FastLanguageModel.for_inference(model)

test_problems = [
    "A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?",
    "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
]

for problem in test_problems:
    messages = [
        {"role": "system", "content": "You are a helpful reasoning assistant. Think through problems step by step before answering."},
        {"role": "user", "content": problem},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to("cuda")
    outputs = model.generate(inputs, max_new_tokens=1024, temperature=0.7)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Problem: {problem}")
    print(f"Response: {response[:500]}")
    print()

In [ ]:
# Step 10: Save merged model and push to HuggingFace
# Save merged 16-bit for HuggingFace
model.save_pretrained_merged(
    "model-merged",
    tokenizer,
    save_method="merged_16bit",
)

# Push merged model to HuggingFace
model.push_to_hub_merged(
    "bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled",
    tokenizer,
    save_method="merged_16bit",
    token=userdata.get('HF_TOKEN'),
)

In [ ]:
# Step 11: Export GGUF for local inference via Ollama/llama.cpp
model.push_to_hub_gguf(
    "bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled",
    tokenizer,
    quantization_method=["q4_k_m", "q8_0"],
    token=userdata.get('HF_TOKEN'),
)
print("GGUF files pushed to HuggingFace!")

## Optional: GRPO Reinforcement Learning

If SFT results look good, continue with GRPO to teach the model to reason *correctly*,
not just in the right style. See the GRPO notebook or the Unsloth GRPO guide:
https://unsloth.ai/blog/r1-reasoning